In [ ]:
!pip install torchreid
!pip install opencv-python-headless
!pip install ultralytics

In [ ]:
import cv2
import torch
import numpy as np
import os
from PIL import Image
from google.colab.patches import cv2_imshow
from google.colab import files, drive
from IPython.display import display, HTML
from base64 import b64encode
import torchreid
from torchvision import transforms

In [ ]:
# Класс для хранения и сравнения фич людей
class PersonGallery:
    """Класс для хранения и сравнения фич людей"""
    def __init__(self, similarity_threshold=0.7):
        self.gallery = {}
        self.next_id = 0
        self.similarity_threshold = similarity_threshold

    def add_person(self, features):
        """Добавление нового человека в галерею"""
        person_id = self.next_id
        self.gallery[person_id] = features
        self.next_id += 1
        return person_id

    def find_similar(self, query_features):
        """Поиск похожего человека в галерее"""
        if not self.gallery:
            return None, 0

        best_similarity = -1
        best_id = None

        for person_id, gallery_features in self.gallery.items():
            similarity = self.cosine_similarity(query_features, gallery_features)
            if similarity > best_similarity:
                best_similarity = similarity
                best_id = person_id

        if best_similarity > self.similarity_threshold:
            return best_id, best_similarity
        else:
            return None, best_similarity

    def cosine_similarity(self, features1, features2):
        """Вычисление косинусной схожести"""
        features1 = features1.flatten()
        features2 = features2.flatten()
        return np.dot(features1, features2) / (
            np.linalg.norm(features1) * np.linalg.norm(features2)
        )

In [ ]:
# Load trained model

def load_trained_model(model_path, num_classes=751):
    """Load re-ID model from clean state_dict"""
    model = torchreid.models.build_model(
        name="resnet50",
        num_classes=num_classes,
        loss="softmax",
        pretrained=False
    )

    state_dict = torch.load(model_path, map_location='cpu')

    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        new_state_dict[name] = v

    model.load_state_dict(new_state_dict)
    model = model.cuda()
    model.eval()

    print("Model loaded")
    return model


In [5]:
# Предобработка изображения
def preprocess_image(image, height=256, width=128):
    """Предобработка изображения для модели re-ID"""
    transform = transforms.Compose([
        transforms.Resize((height, width)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    if isinstance(image, np.ndarray):
        image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    return transform(image).unsqueeze(0)

# Извлечение фич
def extract_features(model, image_tensor):
    """Извлечение features из изображения"""
    with torch.no_grad():
        features = model(image_tensor.cuda())
    return features.cpu().numpy()

# Функция анализа одного изображения
def analyze_single_image(image_path, model_reid, model_yolo):
    """Анализ отдельного изображения"""
    if not os.path.exists(image_path):
        print("Изображение не найдено!")
        return None

    # Загружаем изображение
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Детекция людей
    results = model_yolo(image_rgb)
    boxes = results[0].boxes

    # Создаем галерею для хранения фич
    gallery = PersonGallery(similarity_threshold=0.7)

    people_detected = 0

    if boxes is not None:
        for box in boxes:
            if int(box.cls) == 0:  # person class
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x, y, w, h = int(x1), int(y1), int(x2-x1), int(y2-y1)
                conf = box.conf[0].cpu().numpy()

                # Фильтр по уверенности
                if conf < 0.5:
                    continue

                # Вырезаем область с человеком
                person_roi = image[y:y+h, x:x+w]

                if person_roi.size > 0:
                    try:
                        # Извлекаем фичи
                        processed_image = preprocess_image(person_roi)
                        features = extract_features(model_reid, processed_image)
                        features = features / np.linalg.norm(features)  # нормализуем

                        # Ищем в галерее
                        result = gallery.find_similar(features)
                        if result is None:
                            person_id = None
                            similarity = 0
                        else:
                            person_id, similarity = result

                        if person_id is None:
                            # Новый человек
                            person_id = gallery.add_person(features)
                            color = (0, 255, 0)  # зеленый - новый
                            label = f"ID:{person_id} (new)"
                        else:
                            # Известный человек
                            color = (255, 0, 0)  # синий - известный
                            label = f"ID:{person_id} ({similarity:.2f})"

                        # Рисуем bounding box и подпись
                        cv2.rectangle(image, (x, y), (x+w, y+h), color, 3)
                        cv2.putText(image, label, (x, y-10),
                                   cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

                        people_detected += 1

                    except Exception as e:
                        print(f"Ошибка обработки человека: {e}")
                        continue

    print(f"Обнаружено людей: {people_detected}")

    # Конвертируем обратно в RGB для отображения
    result_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return result_image



In [ ]:
# Основная функция
def main():
    # 1. Загружаем модель re-ID
    print("=== ЗАГРУЗКА МОДЕЛИ RE-ID ===")
    model_path = 'models/resnet50_market1501_clean.pth'  # clean inference weights
    model_reid = load_trained_model(model_path)

    # 2. Загружаем YOLO для детекции людей
    print("\n=== ЗАГРУЗКА ДЕТЕКТОРА YOLO ===")
    from ultralytics import YOLO
    model_yolo = YOLO('yolov8n.pt')

    # 3. Загружаем изображение
    print("\n=== ЗАГРУЗКА ИЗОБРАЖЕНИЯ ===")
    print("Пожалуйста, загрузите изображение для анализа:")
    uploaded = files.upload()

    if not uploaded:
        print("Изображение не загружено!")
        return

    image_path = list(uploaded.keys())[0]
    print(f"Загружено изображение: {image_path}")

    # 4. Обрабатываем изображение
    print("\n=== ОБРАБОТКА ИЗОБРАЖЕНИЯ ===")
    result_image = analyze_single_image(image_path, model_reid, model_yolo)

    # 5. Сохраняем и показываем результат
    print("\n=== РЕЗУЛЬТАТ ===")
    if result_image is not None:
        # Сохраняем обработанное изображение
        output_path = "processed_image.jpg"
        cv2.imwrite(output_path, cv2.cvtColor(result_image, cv2.COLOR_RGB2BGR))

        # Показываем оригинал и результат
        print("Оригинальное изображение:")
        original_image = Image.open(image_path)
        display(original_image)

        print("\nОбработанное изображение (с детекцией и re-ID):")
        display(Image.fromarray(result_image))

        files.download(output_path)
    else:
        print("Не удалось обработать изображение!")

# Запуск
if __name__ == "__main__":
    main()
